In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

dataDir = Path("data")
csvFile = dataDir / "pp-2020.csv"


columns = [
    "transaction_id",
    "price",
    "date_of_transfer",
    "postcode",
    "property_type",
    "new_build",
    "duration",
    "paon",
    "saon",
    "street",
    "locality",
    "town_city",
    "district",
    "county",
    "ppd_category",
    "record_status",
]

allYears = pd.read_csv(csvFile, header=None, names=columns)

# Clean types
allYears["date_of_transfer"] = pd.to_datetime(allYears["date_of_transfer"])
allYears["price"] = pd.to_numeric(allYears["price"])

allYears.head()

,transaction_id,price,date_of_transfer,postcode,property_type,new_build,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
0,{BC8936BB-BC2C-0E2C-E053-6C04A8C0DBF4},103250,2020-01-28,WV12 5AB,F,Y,L,"LICHFIELD HOUSE, 232",FLAT 23,LICHFIELD ROAD,NaN,WILLENHALL,WALSALL,WEST MIDLANDS,A,A
1,{BC8936BB-BC2F-0E2C-E053-6C04A8C0DBF4},176500,2020-11-20,B32 4DA,T,N,F,83,NaN,KITWELL LANE,NaN,BIRMINGHAM,BIRMINGHAM,WEST MIDLANDS,A,A
2,{BC8936BB-BC30-0E2C-E053-6C04A8C0DBF4},97500,2020-10-29,WV6 9NY,F,N,L,WILLOWDALE GRANGE,16,ALDERSLEY ROAD,NaN,WOLVERHAMPTON,WOLVERHAMPTON,WEST MIDLANDS,A,A
3,{BC8936BB-BC31-0E2C-E053-6C04A8C0DBF4},180000,2020-02-04,DY2 9ET,S,N,F,124,NaN,NORTHFIELD ROAD,NaN,DUDLEY,DUDLEY,WEST MIDLANDS,A,A
4,{BC8936BB-BC32-0E2C-E053-6C04A8C0DBF4},196000,2020-03-09,CV6 1HS,S,N,F,135,NaN,MOSELEY AVENUE,NaN,COVENTRY,COVENTRY,WEST MIDLANDS,B,A


In [60]:
allYears.shape

(896568, 16)

In [61]:
allYears.isna().sum().sort_values(ascending=False).head()

saon             783065
locality         545171
street            18035
postcode           3277
property_type         0
dtype: int64

In [62]:
allYears.describe()

,price,date_of_transfer
count,8.965680e+05,896568
mean,3.777048e+05,2020-07-24 20:36:38.720453888
min,1.000000e+00,2020-01-01 00:00:00
25%,1.575000e+05,2020-04-03 00:00:00
50%,2.499950e+05,2020-08-14 00:00:00
75%,3.850000e+05,2020-10-30 00:00:00
max,4.000000e+08,2020-12-31 00:00:00
std,1.712974e+06,NaN


In [63]:
cleanData = allYears[allYears["ppd_category"] == "A"].copy()

In [64]:
cleanData.head()

,transaction_id,price,date_of_transfer,postcode,property_type,new_build,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
0,{BC8936BB-BC2C-0E2C-E053-6C04A8C0DBF4},103250,2020-01-28,WV12 5AB,F,Y,L,"LICHFIELD HOUSE, 232",FLAT 23,LICHFIELD ROAD,NaN,WILLENHALL,WALSALL,WEST MIDLANDS,A,A
1,{BC8936BB-BC2F-0E2C-E053-6C04A8C0DBF4},176500,2020-11-20,B32 4DA,T,N,F,83,NaN,KITWELL LANE,NaN,BIRMINGHAM,BIRMINGHAM,WEST MIDLANDS,A,A
2,{BC8936BB-BC30-0E2C-E053-6C04A8C0DBF4},97500,2020-10-29,WV6 9NY,F,N,L,WILLOWDALE GRANGE,16,ALDERSLEY ROAD,NaN,WOLVERHAMPTON,WOLVERHAMPTON,WEST MIDLANDS,A,A
3,{BC8936BB-BC31-0E2C-E053-6C04A8C0DBF4},180000,2020-02-04,DY2 9ET,S,N,F,124,NaN,NORTHFIELD ROAD,NaN,DUDLEY,DUDLEY,WEST MIDLANDS,A,A
5,{BC8936BB-BC33-0E2C-E053-6C04A8C0DBF4},140000,2020-02-07,CV6 1GT,T,N,L,23,NaN,HAYNESTONE ROAD,NaN,COVENTRY,COVENTRY,WEST MIDLANDS,A,A


In [65]:
columnsToDrop = [
    "transaction_id",
    "paon",
    "saon",
    "street",
    "locality",
    "record_status",
    "county",
    "ppd_category"
]

cleanData.drop(columns=columnsToDrop, inplace=True)

In [66]:
cleanData.head()

,price,date_of_transfer,postcode,property_type,new_build,duration,town_city,district
0,103250,2020-01-28,WV12 5AB,F,Y,L,WILLENHALL,WALSALL
1,176500,2020-11-20,B32 4DA,T,N,F,BIRMINGHAM,BIRMINGHAM
2,97500,2020-10-29,WV6 9NY,F,N,L,WOLVERHAMPTON,WOLVERHAMPTON
3,180000,2020-02-04,DY2 9ET,S,N,F,DUDLEY,DUDLEY
5,140000,2020-02-07,CV6 1GT,T,N,L,COVENTRY,COVENTRY


In [67]:
cleanData.isna().sum().sort_values(ascending=False).head()

postcode            154
price                 0
date_of_transfer      0
property_type         0
new_build             0
dtype: int64

In [68]:
cleanData.describe()

,price,date_of_transfer
count,7.524740e+05,752474
mean,3.271443e+05,2020-07-26 20:05:05.233137664
min,1.000000e+00,2020-01-01 00:00:00
25%,1.690500e+05,2020-04-09 00:00:00
50%,2.575000e+05,2020-08-19 00:00:00
75%,3.900000e+05,2020-10-30 00:00:00
max,5.336000e+07,2020-12-31 00:00:00
std,3.574127e+05,NaN


In [69]:
cleanData = cleanData.dropna(subset=["postcode"])

In [70]:
cleanData.isna().sum().sort_values(ascending=False).head()

price               0
date_of_transfer    0
postcode            0
property_type       0
new_build           0
dtype: int64

In [71]:
cleanData = cleanData[(cleanData["price"] >= 90_000) & (cleanData["price"] <= 5_000_000)]

In [72]:
cleanData.describe()

,price,date_of_transfer
count,7.150210e+05,715021
mean,3.357753e+05,2020-07-27 20:08:43.359173888
min,9.000000e+04,2020-01-01 00:00:00
25%,1.800000e+05,2020-04-14 00:00:00
50%,2.674990e+05,2020-08-20 00:00:00
75%,3.999500e+05,2020-11-02 00:00:00
max,5.000000e+06,2020-12-31 00:00:00
std,2.733817e+05,NaN


In [73]:
cleanData["year"] = cleanData["date_of_transfer"].dt.year
cleanData["month"] = cleanData["date_of_transfer"].dt.month

In [74]:
cleanData["postcode"] = cleanData["postcode"].str.split().str[0]

In [75]:
cleanData.head()

,price,date_of_transfer,postcode,property_type,new_build,duration,town_city,district,year,month
0,103250,2020-01-28,WV12,F,Y,L,WILLENHALL,WALSALL,2020,1
1,176500,2020-11-20,B32,T,N,F,BIRMINGHAM,BIRMINGHAM,2020,11
2,97500,2020-10-29,WV6,F,N,L,WOLVERHAMPTON,WOLVERHAMPTON,2020,10
3,180000,2020-02-04,DY2,S,N,F,DUDLEY,DUDLEY,2020,2
5,140000,2020-02-07,CV6,T,N,L,COVENTRY,COVENTRY,2020,2


In [76]:
categoricalColumns = [
    "property_type",
    "new_build",
    "duration",
    "postcode",
    "town_city",
    "district",
]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded = encoder.fit_transform(cleanData[categoricalColumns])

encodedData = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categoricalColumns),
    index=cleanData.index
)

cleanData = pd.concat(
    [cleanData.drop(columns=categoricalColumns), encodedData],
    axis=1
)

In [77]:
cleanData.head()

,price,date_of_transfer,year,month,property_type_D,property_type_F,property_type_S,property_type_T,new_build_N,new_build_Y,duration_F,duration_L,postcode_AL1,postcode_AL10,postcode_AL2,postcode_AL3,postcode_AL4,postcode_AL5,postcode_AL6,postcode_AL7,postcode_AL8,postcode_AL9,postcode_B1,postcode_B10,postcode_B11,postcode_B12,postcode_B13,postcode_B14,postcode_B15,postcode_B16,postcode_B17,postcode_B18,postcode_B19,postcode_B2,postcode_B20,postcode_B21,postcode_B23,postcode_B24,postcode_B25,postcode_B26,...,district_TRAFFORD,district_TUNBRIDGE WELLS,district_UTTLESFORD,district_VALE OF WHITE HORSE,district_WAKEFIELD,district_WALSALL,district_WALTHAM FOREST,district_WANDSWORTH,district_WARRINGTON,district_WARWICK,district_WATFORD,district_WAVERLEY,district_WEALDEN,district_WELLINGBOROUGH,district_WELWYN HATFIELD,district_WEST BERKSHIRE,district_WEST DEVON,district_WEST LANCASHIRE,district_WEST LINDSEY,district_WEST NORTHAMPTONSHIRE,district_WEST OXFORDSHIRE,district_WEST SUFFOLK,district_WESTMORLAND AND FURNESS,district_WIGAN,district_WILTSHIRE,district_WINCHESTER,district_WINDSOR AND MAIDENHEAD,district_WIRRAL,district_WOKING,district_WOKINGHAM,district_WOLVERHAMPTON,district_WORCESTER,district_WORTHING,district_WREKIN,district_WREXHAM,district_WYCHAVON,district_WYCOMBE,district_WYRE,district_WYRE FOREST,district_YORK
0,103250,2020-01-28,2020,1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,176500,2020-11-20,2020,11,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,97500,2020-10-29,2020,10,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,180000,2020-02-04,2020,2,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,140000,2020-02-07,2020,2,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
target = "price"

y = cleanData[target]
X = cleanData.drop(columns=[target])

In [ ]:
XTrain, XTest, yTrain, yTest = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

XTrain.shape, XTest.shape